# MLP N Digit Addition

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import random
import statistics

## Config

Set N_DIGITS to any positive integer

In [2]:
N_DIGITS   = 25          # ← change this to run on any number of digits
SEED       = 42         # fixed seed for reproducibility; set None for random

if SEED is not None:
    torch.manual_seed(SEED)
    random.seed(SEED)

print(f'Config: N_DIGITS={N_DIGITS}  seed={SEED}')

Config: N_DIGITS=25  seed=42


## Creates the Dataset

In [3]:
rows = []
for a in range(10):
    for b in range(10):
        for cin in range(2):
            x = torch.zeros(21)
            x[a]      = 1.0           # one-hot encode digit a
            x[10 + b] = 1.0           # one-hot encode digit b
            x[20]     = float(cin)    # carry-in as a scalar feature
            rows.append((x, (a + b + cin) % 10, (a + b + cin) // 10))

X       = torch.stack([r[0] for r in rows])           # (200, 21)
Y_ones  = torch.tensor([r[1] for r in rows])          # ones digit of column sum
Y_carry = torch.tensor([r[2] for r in rows])          # carry-out
N       = len(X)                                       # 200 training examples

print('Dataset created')

Dataset created


## Architecture

input (21) → hidden layer 1 (64) (frozen in Model C) → hidden layer 2 (64) → ones head (10 classes) and carry head (2 classes)


In [4]:
class MultiDigitMLP(nn.Module):
    """Processes one digit column: one-hot(a) + one-hot(b) + carry_in."""
    def __init__(self, freeze_layer1=False):
        super().__init__()
        self.hidden1    = nn.Linear(21, 64)
        self.hidden2    = nn.Linear(64, 64)
        self.ones_head  = nn.Linear(64, 10)
        self.carry_head = nn.Linear(64,  2)

        #locks hidden1 for Model C
        if freeze_layer1:
            for p in self.hidden1.parameters():
                p.requires_grad_(False)

    def forward(self, x):
        h1 = self.hidden1(x).clamp(min=0)
        h2 = self.hidden2(h1).clamp(min=0)
        return self.ones_head(h2), self.carry_head(h2)

def accuracy(logits, targets):
    return (logits.argmax(1) == targets).sum().item()

print('Architecture defined')

Architecture defined


## Helper
trains the model and takes the heads you want to be part of the training as input parameters

In [5]:
def train(model, heads='both', max_epochs=50000, lr=1e-3):
    params = [p for p in model.parameters() if p.requires_grad]
    opt    = optim.Adam(params, lr=lr)
    ce     = nn.CrossEntropyLoss()

    loss_history = []   # (epoch, loss_val) pairs collected every 1000 epochs

    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        opt.zero_grad()
        lo, lc = model(X)
        loss = ce(lo, Y_ones)
        if heads == 'both':
            loss = loss + ce(lc, Y_carry)
        loss.backward()
        opt.step()

        co = accuracy(lo, Y_ones)
        cc = accuracy(lc, Y_carry)

        if epoch % 1000 == 0:
            loss_history.append((epoch, loss.item()))  # record for analysis
        if epoch % 5000 == 0:
            print(f'  epoch {epoch:5d}  loss={loss.item():.4f}  ones={co}/{N}  carry={cc}/{N}')

        done = (co == N) if heads == 'ones' else (co == N and cc == N)
        if done:
            loss_history.append((epoch, loss.item()))
            print(f'  100% at epoch {epoch}')
            break

    elapsed = time.time() - t0
    return elapsed, epoch, loss_history

print('Training helper defined')

Training helper defined


## N-Digit Stacking Helper

Two calls to a `MultiDigitMLP` per column, ripple-carry from least-significant to most-significant.
Works for any `N_DIGITS` — the loop replaces the hard-coded two-pass logic from the prior notebook.

In [6]:
def add_n_digit(net, a_val, b_val, n=N_DIGITS):
    """Add two n-digit numbers by stacking the column MLP n times with ripple-carry."""
    digits_a = [(a_val // (10 ** i)) % 10 for i in range(n)]  # LSB-first
    digits_b = [(b_val // (10 ** i)) % 10 for i in range(n)]

    result_digits = []
    carry = 0
    with torch.no_grad():
        for col in range(n):
            x = torch.zeros(21)
            x[digits_a[col]]      = 1.0          # one-hot encode digit a
            x[10 + digits_b[col]] = 1.0          # one-hot encode digit b
            x[20]                 = float(carry)  # carry-in as a scalar feature
            lo, lc = net(x.unsqueeze(0))
            result_digits.append(lo.argmax(1).item())
            carry = lc.argmax(1).item()

    # final carry-out becomes the leading digit if non-zero
    if carry:
        result_digits.append(carry)

    return sum(d * (10 ** i) for i, d in enumerate(result_digits))


def exhaustive_check(net, label, n=N_DIGITS):
    """Check all (10^n)^2 pairs and report per-position accuracy."""
    max_val  = 10 ** n
    total    = max_val * max_val
    errors   = []
    col_errs = [0] * (n + 1)   # per-digit-position error counts (incl. overflow)

    for a in range(max_val):
        for b in range(max_val):
            pred = add_n_digit(net, a, b, n)
            gt   = a + b
            if pred != gt:
                errors.append((a, b, pred, gt))
                # find which digit positions differ
                max_digits = n + 1
                for pos in range(max_digits):
                    pd = (pred // (10 ** pos)) % 10
                    gd = (gt   // (10 ** pos)) % 10
                    if pd != gd:
                        col_errs[pos] += 1

    correct = total - len(errors)
    print(f'{label}')
    print(f'  Overall: {correct}/{total}  ({100*correct/total:.1f}%)')
    if errors:
        print(f'  Errors by digit position (0=ones): {col_errs}')
        print(f'  First 5 errors:')
        for a, b, pred, gt in errors[:5]:
            print(f'    {a} + {b} = {pred}  (expected {gt})')
    return correct

print('Stacking and exhaustive-check helpers defined')

Stacking and exhaustive-check helpers defined


In [7]:
import random

def sample_check(net, label, n=N_DIGITS, num_samples=100000):
    max_val = 10 ** n
    errors = []
    col_errs = [0] * (n + 1)

    for _ in range(num_samples):
        a = random.randrange(max_val)
        b = random.randrange(max_val)

        pred = add_n_digit(net, a, b, n)
        gt = a + b

        if pred != gt:
            errors.append((a, b, pred, gt))
            for pos in range(n + 1):
                pd = (pred // (10 ** pos)) % 10
                gd = (gt // (10 ** pos)) % 10
                if pd != gd:
                    col_errs[pos] += 1

    correct = num_samples - len(errors)
    print(label)
    print(f'  Overall: {correct}/{num_samples} ({100*correct/num_samples:.3f}%)')

    if errors:
        print(f'  Errors by digit position (0=ones): {col_errs}')
        print('  First 5 errors:')
        for a, b, pred, gt in errors[:5]:
            print(f'    {a} + {b} = {pred} (should be {gt})')

    return correct / num_samples


## Model A — ones head only

In [8]:
model_a = MultiDigitMLP()
print('Training Model A (ones only)...')
t_a, ep_a, hist_a = train(model_a, heads='ones')
print(f'Model A training time: {t_a:.2f}s  ({ep_a} epochs)')

model_a.eval()
for p in model_a.parameters():
    p.requires_grad_(False)

Training Model A (ones only)...
  100% at epoch 190
Model A training time: 0.30s  (190 epochs)


## Verify Model A

In [9]:
with torch.no_grad():
    lo, lc = model_a(X)
    print(f'Model A  ones accuracy : {accuracy(lo, Y_ones)}/{N}')
    print(f'Model A  carry accuracy: {accuracy(lc, Y_carry)}/{N}  (untrained)')

Model A  ones accuracy : 200/200
Model A  carry accuracy: 100/200  (untrained)


## Model B — both heads from scratch

In [10]:
model_b = MultiDigitMLP()
print('Training Model B (both heads from scratch)...')
t_b, ep_b, hist_b = train(model_b, heads='both')
print(f'Model B training time: {t_b:.2f}s  ({ep_b} epochs)')

model_b.eval()
for p in model_b.parameters():
    p.requires_grad_(False)

Training Model B (both heads from scratch)...
  100% at epoch 349
Model B training time: 0.67s  (349 epochs)


## Verify Model B

In [11]:
with torch.no_grad():
    lo, lc = model_b(X)
    print(f'Model B  ones accuracy : {accuracy(lo, Y_ones)}/{N}')
    print(f'Model B  carry accuracy: {accuracy(lc, Y_carry)}/{N}')

Model B  ones accuracy : 200/200
Model B  carry accuracy: 200/200


## Model C — fine-tune from Model A with first layer frozen

load_state_dict copies all of Model A's weights. Only hidden2, ones_head, and carry_head receive gradients.

In [12]:
model_c = MultiDigitMLP(freeze_layer1=True)
model_c.load_state_dict(model_a.state_dict())   # warm-start from Model A

# Ensure everything except h1 is trainable
for name, p in model_c.named_parameters():
    if 'hidden1' not in name:
        p.requires_grad_(True)

print('Model C: hidden1 frozen, all other layers trainable')
t_c, ep_c, hist_c = train(model_c, heads='both')
print(f'Model C training time: {t_c:.2f}s  ({ep_c} epochs)')
model_c.eval()

Model C: hidden1 frozen, all other layers trainable
  100% at epoch 123
Model C training time: 0.18s  (123 epochs)


MultiDigitMLP(
  (hidden1): Linear(in_features=21, out_features=64, bias=True)
  (hidden2): Linear(in_features=64, out_features=64, bias=True)
  (ones_head): Linear(in_features=64, out_features=10, bias=True)
  (carry_head): Linear(in_features=64, out_features=2, bias=True)
)

## Verify Model C

In [13]:
with torch.no_grad():
    lo, lc = model_c(X)
    print(f'Model C  ones accuracy : {accuracy(lo, Y_ones)}/{N}')
    print(f'Model C  carry accuracy: {accuracy(lc, Y_carry)}/{N}')

Model C  ones accuracy : 200/200
Model C  carry accuracy: 200/200


## Stack for N-digit addition

N calls to a MultiDigitMLP, with the carry output of the ones column feeding into the tens column.

In [14]:
NUM_SAMPLES = 10000
score_a = sample_check(model_a, 'Model A (ones-only train):', n=N_DIGITS, num_samples=NUM_SAMPLES)
score_b = sample_check(model_b, 'Model B (both heads, scratch):', n=N_DIGITS, num_samples=NUM_SAMPLES)
score_c = sample_check(model_c, 'Model C (frozen h1, fine-tune):', n=N_DIGITS, num_samples=NUM_SAMPLES)

Model A (ones-only train):
  Overall: 0/10000 (0.000%)
  Errors by digit position (0=ones): [0, 5462, 5060, 4954, 5091, 4979, 4984, 4981, 4953, 5028, 5022, 5047, 5026, 5053, 4990, 4987, 5002, 5013, 4943, 4977, 4908, 5005, 5056, 4976, 5054, 5048]
  First 5 errors:
    483767917028887349000605 + 4736889143851171664249083 = 15220657161980069014350798 (should be 5220657060880059013249688)
    1681633696740388639203664 + 614763976084376907318377 = 12306407673835765647622041 (should be 2296397672824765546522041)
    4229065381222696783759982 + 3846082705536183891070899 = 18176158197869880685830881 (should be 8075148086758880674830881)
    4263935737924575788353782 + 5380999903310648157625642 = 10654935741345224946089434 (should be 9644935641235223945979424)
    125712149878847573513611 + 3088258077097624971850507 = 14214071227976572555474228 (should be 3213970226976472545364118)
Model B (both heads, scratch):
  Overall: 10000/10000 (100.000%)
Model C (frozen h1, fine-tune):
  Overall: 10000/

## Compare Training Times

In [15]:
total_pairs = (10 ** N_DIGITS) ** 2
print('=' * 60)
print(f'Model A  (ones only, from scratch)  : {t_a:.2f}s  {ep_a:5d} epochs  {N_DIGITS}-digit acc: {score_a}/{total_pairs}')
print(f'Model B  (both heads, from scratch) : {t_b:.2f}s  {ep_b:5d} epochs  {N_DIGITS}-digit acc: {score_b}/{total_pairs}')
print(f'Model C  (both heads, frozen h1)    : {t_c:.2f}s  {ep_c:5d} epochs  {N_DIGITS}-digit acc: {score_c}/{total_pairs}')
print('=' * 60)

if t_c < t_b:
    print(f'C was faster than B by {t_b - t_c:.2f}s')
else:
    print(f'B was faster than C by {t_c - t_b:.2f}s')

if t_a + t_c < t_b:
    print(f'A+C together were faster than B by {t_b - (t_a + t_c):.2f}s')
else:
    print(f'B was faster than A+C together by {(t_a + t_c) - t_b:.2f}s')

Model A  (ones only, from scratch)  : 0.30s    190 epochs  25-digit acc: 0.0/100000000000000000000000000000000000000000000000000
Model B  (both heads, from scratch) : 0.67s    349 epochs  25-digit acc: 1.0/100000000000000000000000000000000000000000000000000
Model C  (both heads, frozen h1)    : 0.18s    123 epochs  25-digit acc: 1.0/100000000000000000000000000000000000000000000000000
C was faster than B by 0.49s
A+C together were faster than B by 0.19s


## Loss Curve Analysis

Training captured (epoch, loss) pairs every 1000 steps. Compare how each model's
loss descends which could be useful for spotting whether fine-tuning (Model C) gets an early lead.

In [16]:
def show_loss_curve(history, label):
    if not history:
        print(f'{label}: no history recorded')
        return
    print(f'{label}  ({len(history)} checkpoints)')
    for epoch, loss_val in history[:5]:
        print(f'  epoch {epoch:5d}  loss={loss_val:.4f}')
    if len(history) > 5:
        print(f'  ...')
        epoch, loss_val = history[-1]
        print(f'  epoch {epoch:5d}  loss={loss_val:.4f}  (final)')

show_loss_curve(hist_a, 'Model A')
show_loss_curve(hist_b, 'Model B')
show_loss_curve(hist_c, 'Model C')

Model A  (1 checkpoints)
  epoch   190  loss=0.5682
Model B  (1 checkpoints)
  epoch   349  loss=0.2098
Model C  (1 checkpoints)
  epoch   123  loss=0.3958
